# 34 — Protocolo y piloto: tres PDE y cinco selectores

**Preparado, sin ejecutar.** Este notebook verifica la infraestructura antes de congelar un estudio de 150 corridas. No ejecuta ese estudio: el piloto tiene 15 corridas de 3000 pasos y una semilla exclusiva de desarrollo. Los pasos del piloto no se añaden luego al estudio completo.

Problemas: Burgers, onda y Allen–Cahn. Métodos: Random, Hammersley, Random-R, RAD y PCA + media. Los módulos compartidos evitan duplicar implementaciones en los futuros notebooks 35–37; el 38 agregará archivos con el mismo esquema.

Fuente del protocolo: [QR-DEIM, v4, sección 2.3](https://arxiv.org/html/2501.07700v4#S2.SS3). No es una reproducción exacta: usamos JAX, una variante PCA propia, una secuencia Hammersley desplazada y referencias numéricas propias verificadas por refinamiento.

## 1. Configuración que no se ajustará usando prueba

| Elemento | Piloto | Estudio completo propuesto |
|---|---|---|
| Red | 5 capas ocultas × 64, tanh | Igual |
| Puntos activos | 2000 en todos los métodos | Igual |
| Actualización adaptativa | Cada 1000 pasos | Igual |
| Adam | LR inicial 0.001, coseno con horizonte 100000 | Igual |
| Pasos totales | 3000, sin warmup añadido | 100000 |
| Semillas | 3101 | 3201–3210 reservadas, todavía no utilizadas |
| Validación | 10000 puntos uniformes fijos por semilla; cada 250 pasos | Pendiente de congelación tras piloto |
| Prueba | Rejilla independiente 256×100; solo evaluación final | Igual |

El piloto conserva los primeros 3000 pasos del calendario coseno de 100000, sin comprimirlo. Selección principal de modelo: menor MSE residual de validación, incluyendo el estado inicial; también se guarda el modelo terminal. No se eligen pasos mediante error de solución de prueba. Las referencias no entran en la pérdida. El residuo físico reportado al final usa otro conjunto independiente de validación.

Ejecutar las celdas en orden con el kernel Python del proyecto. Las celdas 4 y 5 calculan referencias y entrenan respectivamente. También puede ejecutarse todo desde PowerShell con el script indicado al final.

In [ ]:
from pathlib import Path
import sys, os, json
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ.setdefault('MPLBACKEND', 'Agg')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pinn_pca.benchmark_pilot import prepare, references, run_pilot, report
CONFIG_PATH = ROOT / 'pca_collocation/configs/benchmark_protocol_pilot.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
OUT = prepare(ROOT, config)
print(json.dumps(config, indent=2))
print('Resultados:', OUT)


## 2. Las ecuaciones y las restricciones exactas

Dominio común: $x\in[-1,1]$, $t\in[0,1]$.

| Problema | PDE | Estado inicial | Frontera |
|---|---|---|---|
| Burgers | $u_t+uu_x-(0.01/\pi)u_{xx}=0$ | $-\sin(\pi x)$ | $u(\pm1,t)=0$ |
| Onda | $u_{tt}-4u_{xx}=0$ | $\sin(\pi x)+0.5\sin(4\pi x)$, $u_t(x,0)=0$ | $u(\pm1,t)=0$ |
| Allen–Cahn | $u_t-0.0001u_{xx}-5(u^3-u)=0$ | $x^2\cos(\pi x)$ | $u(\pm1,t)=-1$ |

**Signo Allen–Cahn:** se implementa literalmente el signo positivo de $5(u^3-u)$ que aparece en la sección 3.3 del artículo, no se sustituye por la formulación usual $5(u-u^3)$. La referencia numérica usa el mismo signo. No se importan datos AC de otro problema. Revisar este punto antes del estudio completo si se pretende reproducir una implementación externa.

Transformación de salida: $u=u_0(x)+t(1-x^2)N_\theta(x,t)$ para Burgers y Allen–Cahn. Para onda se usa $t^2(1-x^2)$: así también se impone la velocidad inicial nula. El residuo se deriva de la salida transformada, mediante diferenciación automática. No hay etiquetas supervisadas en el entrenamiento.

## 3. Qué cambia entre selectores

- **Random:** conjunto uniforme fijo desde el principio.
- **Hammersley:** secuencia base 2 fija, con desplazamiento módulo uno por semilla. Es una variante aleatorizada para permitir repeticiones; no se presenta como la implementación exacta del artículo.
- **Random-R:** reemplazo completo uniforme en pasos 1000 y 2000 del piloto.
- **RAD:** en esos mismos pasos evalúa 10000 candidatos uniformes nuevos; selecciona sin reemplazo con probabilidad proporcional a |residuo|² (k=2, c=0). Si todos son cero usa uniforme.
- **PCA + media:** 4096 candidatos uniformes fijos y 20 snapshots cada 50 pasos durante el intervalo. PCA retiene 95 % de varianza centrada; se añade y ortogonaliza la media, que cuenta en el rango. QR directo selecciona k puntos; el resto hasta 2000 es uniforme. No hay ampliación log-det ni poda.

Random, Random-R, RAD y PCA empiezan exactamente con los mismos puntos uniformes y pesos por semilla. Hammersley comparte los pesos pero tiene su propio conjunto fijo. No hay un entrenamiento uniforme compartido extra; durante los primeros 1000 pasos PCA recoge su primera historia.

**Dilución que queremos medir:** con veinte snapshots, esta construcción puede aportar como máximo veinte direcciones independientes (incluida la media), es decir, aproximadamente 1 % de los 2000 puntos. Este piloto registra el rango real y su fracción; no modifica automáticamente el presupuesto guiado. Si el componente resulta demasiado pequeño, habrá que definir otro control antes de congelar el estudio. No se elegirá esa modificación por el ganador de este piloto.

## 4. Preparar y verificar las referencias

Onda usa la solución analítica: sin(πx) cos(2πt) + 0.5 sin(4πx) cos(8πt). Burgers y Allen–Cahn se resuelven mediante diferencias centrales espaciales y BDF, con 512 y 1024 intervalos. Se exige diferencia relativa L2 entre las dos referencias menor a 2 % sobre la rejilla de prueba. Es un control de consistencia, no una cota rigurosa de error ni la solución de referencia original del artículo.

Si falla, el programa se detiene antes de entrenar; no relaja la tolerancia por sí solo. Los archivos y auditorías quedan en references/. Cambiar configuración exige un run_id nuevo.

In [ ]:
reference_folder = references(ROOT, config)
print((reference_folder / 'audit.json').read_text(encoding='utf-8'))


## 5. Ejecutar únicamente el piloto

Esta celda ejecuta 15 corridas de 3000 pasos; puede tardar considerablemente en CPU, especialmente para onda. En consola aparece progreso cada 250 pasos. Se guardan trazas incrementalmente, puntos, matrices de historia, pesos terminales y mejores pesos de validación. El estado terminal también incluye los momentos de Adam.

Se verifican IC, BC y velocidad inicial de onda antes y después de cada corrida; se verifica tamaño y unicidad de todos los lotes. Las corridas completas se omiten al repetir el comando. Las incompletas se reproducen desde el principio; no se restaura Adam a mitad de corrida. No ejecutar simultáneamente el script y esta celda.

In [ ]:
OUT = run_pilot(ROOT, config)
print('Piloto completo:', OUT)


## 6. Leer los resultados sin volver a entrenar

Los errores del piloto son diagnósticos de funcionamiento, sin intervalos confirmatorios ni selección de ganador. Las figuras comparan errores y ubicación de puntos. Revisar especialmente el rango PCA y la fracción de puntos guiados. Un umbral de advertencia del 5 % es solo una señal para revisar la dilución, no un criterio estadístico de aprobación.

In [ ]:
from IPython.display import Image, display
import pandas as pd
report(ROOT, config)  # Falla si falta una corrida; nunca entrena.
display(pd.read_csv(OUT / 'metrics.csv'))
display(pd.read_csv(OUT / 'pca_rank_diagnostics.csv'))
print((OUT / 'pilot_review.json').read_text(encoding='utf-8'))
for name in ('pilot_errors.png', 'pilot_points.png'):
    display(Image(filename=str(OUT / name), format='png', embed=True))


## 7. Cómo decidir el paso siguiente

Antes de implementar los notebooks 35–37: revisar referencias, restricciones, estabilidad numérica, fracción guiada y recursos observados. Congelar el significado de PCA + media y el signo Allen–Cahn. Un piloto corto no establece convergencia ni superioridad.

Después, cada problema tendrá su notebook de 50 corridas y los mismos campos de métricas: problem, method, seed, checkpoint, step, relative_l2, rmse, independent_residual_rms, pipeline_seconds. El notebook 38 agregará archivos completos sin volver a entrenar y sin promediar RMSE de problemas diferentes. Las semillas de confirmación reservadas no se ejecutan aquí.

## 8. Ejecución desde terminal y archivos persistentes

Desde la raíz: `python -u scripts/run_benchmark_protocol_pilot.py`. Guarda consola en execution.log, estado en status.json, configuración en config.json, posibles errores en last_error.txt y archivos por problema/método/semilla dentro de pca_collocation/results/benchmark_pilot/three_pdes_five_selectors_pilot_v1/.

Para regenerar solo tablas y figuras: `python scripts/run_benchmark_protocol_pilot.py --report-only`. No lanza entrenamientos si faltan archivos: se detiene. El script crea al final 34_protocol_pilot_results.ipynb, con PNG incrustados y listo para revisar aquí. Este notebook de protocolo permanece sin ejecución al generarlo; si lo ejecutas manualmente, guarda el notebook para conservar sus salidas.

JupyterLab requiere el extra [notebooks] del proyecto. El script de terminal usa el entorno actual de investigación y evita depender de un servidor Jupyter. La importación de módulos no ejecuta entrenamiento.